In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

porto_seguro_safe_driver_prediction_path = kagglehub.competition_download('porto-seguro-safe-driver-prediction')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/porto-seguro-safe-driver-prediction/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

# 피처 엔지니어링

## 데이터 합치기

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.drop('target', axis=1)

In [ ]:
all_features = all_data.columns
all_data

,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,2,2,5,1,0,0,1,0,0,0,...,9,1,5,8,0,1,1,0,0,1
1,1,1,7,0,0,0,0,1,0,0,...,3,1,1,9,0,1,1,0,1,0
2,5,4,9,1,0,0,0,1,0,0,...,4,2,7,7,0,1,1,0,1,0
3,0,1,2,0,0,1,0,0,0,0,...,2,2,4,9,0,0,0,0,0,0
4,0,2,0,1,0,1,0,0,0,0,...,3,1,1,3,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1488023,0,1,6,0,0,0,1,0,0,0,...,4,2,3,4,0,1,0,0,1,0
1488024,5,3,5,1,0,0,0,1,0,0,...,6,2,2,11,0,0,1,1,0,0
1488025,0,1,5,0,0,1,0,0,0,0,...,5,2,2,11,0,1,1,0,0,0
1488026,6,1,5,1,0,0,0,0,1,0,...,1,1,2,7,1,1,0,0,0,0


## 명목형 피처 원-핫 인코딩

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
cat_features = [feature for feature in all_features if 'cat' in feature]

In [ ]:
onehot_encoder = OneHotEncoder()
encoded_cat_matrix = onehot_encoder.fit_transform(all_data[cat_features])
encoded_cat_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20832392 stored elements and shape (1488028, 184)>

## 필요 없는 피처 제거

In [ ]:
drop_features = ['ps_ind_14', 'ps_ind_10_bin', 'ps_ind_11_bin',
                 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_car_14']

In [ ]:
remaining_features = [feature for feature in all_features
                      if ('cat' not in feature and
                          'calc' not in drop_features)]

In [ ]:
from scipy import sparse

In [ ]:
all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data[remaining_features]),
                               encoded_cat_matrix],
                              format='csr')

## 데이터 나누기

In [ ]:
num_train = len(train)

In [ ]:
X = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

In [ ]:
y = train['target'].values

# 평가지표 계산 함수 작성

## 정규화 지니계수 계산 함수

In [ ]:
def eval_gini(y_true, y_pred):
    # 실제값과 예측값의 크기가 서로 같은지 확인 (값이 다르면 오류 발생)
    assert y_true.shape == y_pred.shape

    n_samples = y_true.shape[0]
    L_mid = np.linspace(1/n_samples, 1, n_samples)

    # 1) 예측값에 대한 지니계수
    pred_order = y_true[y_pred.argsort()]
    L_pred = np.cumsum(pred_order) / np.sum(pred_order)
    G_pred = np.sum(L_mid - L_pred)

    # 2) 예측이 완벽할 때 지니계수
    true_order = y_true[y_true.argsort()]
    L_true = np.cumsum(true_order) / np.sum(true_order)
    G_true = np.sum(L_mid - L_true)

    # 정규화된 지니계수
    return G_pred / G_true

In [ ]:
# 모델 훈련 시 검증 파라미터에 전달하기 위한 함수
def gini(preds, dtrain):
    labels = dtrain.get_label()
    return 'gini', eval_gini(labels, preds), True
        # 평가지표 이름, 평가 점수, 평가 점수가 높을수록 좋은지 여부

# 모델 훈련 및 성능 검증

## OOF 방식으로 LightGBM 훈련

### OOF 검증 방식

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
# 층화 K 폴드 교차 검증기
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1991)

In [ ]:
params = {'objective': 'binary',
          'learning_rate': 0.1,
          'force_row_wise': True,
          'random_state': 0}

In [ ]:
# OOF 방식으로 훈련된 모델로 검증 데이터 타깃값을 예측한 확률을 담을 1차원 배열
oof_val_preds = np.zeros(X.shape[0])

# OOF 방식으로 훈련된 모델로 테스트 데이터 타깃값을 예측한 확률을 담을 1차원 배열
oof_test_preds = np.zeros(X_test.shape[0])

### LightGBM 모델 훈련

In [ ]:
import lightgbm as lgb

In [ ]:
# OOF 방식으로 모델 훈련, 검증, 예측
for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    print('#'*40, f'폴드 {idx+1} / 폴드 {folds.n_splits}', '#'*40)

    # 훈련용 데이터, 검증용 데이터 설정
    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]

    # LightBGM 전용 데이터셋 생성
    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_valid, y_valid)

    # LightBGM 모델 훈련
    lgb_model = lgb.train(params=params,            # 훈령용 하이퍼파라미터
                          train_set=dtrain,         # 훈련 데이터셋
                          num_boost_round=1000,     # 부스팅 반복 횟수
                          valid_sets=dvalid,        # 성능 평가용 검증 데이터셋
                          feval=gini,               # 검증용 평가지표
                          # early_stopping_rounds=100,# 조기종료 조건
                          # verbose_eval=100,         # 100번째마다 점수 출력
                          callbacks=[
                              lgb.early_stopping(stopping_rounds=100),
                              lgb.log_evaluation(100)
                        ])

    # 테스트 데이터를 활용해 OOF 예측
    oof_test_preds += lgb_model.predict(X_test)/folds.n_splits
    # 모델 성능 평가를 위한 검증 데이터 타깃값 예측
    oof_val_preds[valid_idx] += lgb_model.predict(X_valid)

    # 검증 데이터 예측 확률에 대한 정규화 지니계수
    gini_score = eval_gini(y_valid, oof_val_preds[valid_idx])
    print(f'폴드 {idx+1} 지니계수: {gini_score}/n')

######################################## 폴드 1 / 폴드 5 ########################################
[LightGBM] [Info] Number of positive: 17355, number of negative: 458814
[LightGBM] [Info] Total Bins 1545
[LightGBM] [Info] Number of data points in the train set: 476169, number of used features: 226
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764
Training until validation scores don't improve for 3 rounds
Early stopping, best iteration is:
[49]	valid_0's binary_logloss: 0.152037	valid_0's gini: 0.281467
폴드 1 지니계수: 0.28146722394514995/n
######################################## 폴드 2 / 폴드 5 ########################################
[LightGBM] [Info] Number of positive: 17355, number of negative: 458814
[LightGBM] [Info] Total Bins 1542
[LightGBM] [Info] Number of data points in the train set: 476169, number of used features: 226
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[Li

In [ ]:
print('OOF 검증 데이터 지니계수: ', eval_gini(y, oof_val_preds))

OOF 검증 데이터 지니계수:  0.27427239017596794


# 예측 및 결과 제출

In [ ]:
submission['target'] = oof_test_preds
submission.to_csv('submission.csv')

# 성능 개선 1: LightGBM 모델

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/porto-seguro-safe-driver-prediction/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

## 피처 엔지니어링

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.drop('target', axis=1)

In [ ]:
all_features = all_data.columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
cat_features = [feature for feature in all_features if 'cat' in feature]

In [ ]:
onehot_encoder = OneHotEncoder()
encoded_cat_matrix = onehot_encoder.fit_transform(all_data[cat_features])

### 파생 피처 추가

#### 첫 번째: 결측값 피처 (새로운 파생 피처)

In [ ]:
# 데이터 하나당 결측값 개수를 파생 피처로 추가
all_data['num_missing'] = (all_data==-1).sum(axis=1)

In [ ]:
# 명목형 피처, calc 분류의 피처를 제외한 피처
remaining_features = [feature for feature in all_features
                      if ('cat' not in feature and 'calc' not in feature)]

In [ ]:
# num_missing을 remaining_features에 추가
remaining_features.append('num_missing')

#### 두 번째: ind 피처 값을 연결한 피처 (새로운 파생 피처)

In [ ]:
# 분류가 ind인 피처
ind_features = [feature for feature in all_features if 'ind' in feature]

In [ ]:
is_first_feature = True

In [ ]:
for ind_feature in ind_features:
    if is_first_feature:
        all_data['mix_ind'] = all_data[ind_feature].astype(str) + '_'
        is_first_feature = False
    else:
        all_data['mix_ind'] += all_data[ind_feature].astype(str) + '_'

In [ ]:
all_data['mix_ind']

0          2_2_5_1_0_0_1_0_0_0_0_0_0_0_11_0_1_0_
1           1_1_7_0_0_0_0_1_0_0_0_0_0_0_3_0_0_1_
2          5_4_9_1_0_0_0_1_0_0_0_0_0_0_12_1_0_0_
3           0_1_2_0_0_1_0_0_0_0_0_0_0_0_8_1_0_0_
4           0_2_0_1_0_1_0_0_0_0_0_0_0_0_9_1_0_0_
                           ...                  
1488023     0_1_6_0_0_0_1_0_0_0_0_0_0_0_2_0_0_1_
1488024    5_3_5_1_0_0_0_1_0_0_0_0_0_0_11_1_0_0_
1488025     0_1_5_0_0_1_0_0_0_0_0_0_0_0_5_0_0_1_
1488026    6_1_5_1_0_0_0_0_1_0_0_0_0_0_13_1_0_0_
1488027    7_1_4_1_0_0_0_0_1_0_0_0_0_0_12_1_0_0_
Name: mix_ind, Length: 1488028, dtype: object

#### 세 번째: 명목형 피처의 고윳값별 개수 피처 (새로운 파생 피처)

In [ ]:
all_data['ps_ind_02_cat'].value_counts()

ps_ind_02_cat
 1    1079327
 2     309747
 3      70172
 4      28259
-1        523
Name: count, dtype: int64

In [ ]:
all_data['ps_ind_02_cat'].value_counts().to_dict()

{1: 1079327, 2: 309747, 3: 70172, 4: 28259, -1: 523}

In [ ]:
cat_count_features = []
for feature in cat_features+['mix_ind']:
    val_counts_dict = all_data[feature].value_counts().to_dict()
    all_data[f'{feature}_count'] = all_data[feature].apply(lambda x: val_counts_dict[x])
    cat_count_features.append(f'{feature}_count')

In [ ]:
cat_count_features

['ps_ind_02_cat_count',
 'ps_ind_04_cat_count',
 'ps_ind_05_cat_count',
 'ps_car_01_cat_count',
 'ps_car_02_cat_count',
 'ps_car_03_cat_count',
 'ps_car_04_cat_count',
 'ps_car_05_cat_count',
 'ps_car_06_cat_count',
 'ps_car_07_cat_count',
 'ps_car_08_cat_count',
 'ps_car_09_cat_count',
 'ps_car_10_cat_count',
 'ps_car_11_cat_count',
 'mix_ind_count']

### 필요 없는 피처 제거

In [ ]:
from scipy import sparse

In [ ]:
drop_features = ['ps_ind_14', 'ps_ind_10_bin', 'ps_ind_11_bin',
                 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_car_14']

In [ ]:
all_data_remaining = all_data[remaining_features+cat_count_features].drop(drop_features, axis=1)

In [ ]:
all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data_remaining),
                               encoded_cat_matrix],
                              format='csr')

### 데이터 나누기

In [ ]:
num_train = len(train)

In [ ]:
X = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

In [ ]:
y = train['target'].values

## 하이퍼파라미터 최적화

### 데이터셋 준비

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

In [ ]:
# 8:2 비율로 훈련 데이터, 검증 데이터 분리 (베이지안 최적화 수행용)
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      test_size=0.2,
                                                      random_state=0)

In [ ]:
# 베이지안 최적화용 데이터셋
bayes_dtrain = lgb.Dataset(X_train, y_train)
bayes_dvalid = lgb.Dataset(X_valid, y_valid)

### 하이퍼파라미터 범위 설정

In [ ]:
# 베이지안 최적화를 위한 하이퍼파라미터 범위
param_bounds = {'num_leaves': (30, 40),
                'lambda_l1': (0.7, 0.9),
                'lambda_l2': (0.9, 1),
                'feature_fraction': (0.6, 0.7),
                'bagging_fraction': (0.6, 0.9),
                'min_child_samples': (6, 10),
                'min_child_weight': (10, 40),
                'min_data_in_leaf': (6, 50)} # int 범위로 설정

In [ ]:
# 값이 고정된 하이퍼파라미터
fixed_params = {'objective': 'binary',  # 이진분류
                'learning_rate': 0.005, # 학습률
                'bagging_freq': 1,      # 배깅 수행 빈도
                'force_row_wise': True, # 겅고 문구 없애기
                'random_state': 1991}   # random_state 값 고정 => 다시 실행해도 동일한 결과가 나오도록

### (베이지안 최적화용) 평가지표 계산 함수 작성

In [ ]:
# 최적화하려는 평가지표(지니계수) 계산 함수
def eval_function(num_leaves, lambda_l1, lambda_l2, feature_fraction,
                  bagging_fraction, min_child_samples, min_child_weight,
                  min_data_in_leaf):

    # 베이지안 최적화를 수행할 하이퍼파라미터
    params = {'num_leaves': int(round(num_leaves)),
              'lambda_l1': lambda_l1,
              'lambda_l2': lambda_l2,
              'feature_fraction': feature_fraction,
              'bagging_fraction': bagging_fraction,
              'min_child_samples': min_child_samples,
              'min_child_weight': min_child_weight,
              'feature_pre_filter': False,
              'min_data_in_leaf': int(round(min_data_in_leaf))} # min_data_in_leaf 추가 및 int로 변환

    # 고정된 하이퍼파라미터도 추가
    params.update(fixed_params)

    print('하이퍼파라미터: ', params)

    # LightGBM 모델 훈련
    lgb_model = lgb.train(params=params,
                          train_set=bayes_dtrain,
                          num_boost_round=2500,
                          valid_sets=bayes_dvalid,
                          feval=gini,
                          # early_stopping_rounds=300,
                          # verbose_eval=False,
                          callbacks=[
                              lgb.early_stopping(stopping_rounds=300)
                        ])

    # 검증 데이터로 예측 수행
    preds = lgb_model.predict(X_valid)

    # 지니계수 계산
    gini_score = eval_gini(y_valid, preds)
    print(f'지니계수: {gini_score}\n')

    return gini_score

### 최적화 수행

In [ ]:
from bayes_opt import BayesianOptimization

In [ ]:
optimizer = BayesianOptimization(f=eval_function,      # 평가지표 계산 함수
                                 pbounds=param_bounds, # 하이퍼파라미터 범위
                                 random_state=0)

In [ ]:
optimizer.maximize(init_points=3, n_iter=6)

|   iter    |  target   | num_le... | lambda_l1 | lambda_l2 | featur... | baggin... | min_ch... | min_ch... | min_da... |
-------------------------------------------------------------------------------------------------------------------------
하이퍼파라미터:  {'num_leaves': 35, 'lambda_l1': 0.8430378732744839, 'lambda_l2': 0.9602763376071644, 'feature_fraction': 0.6544883182996897, 'bagging_fraction': 0.7270964398016714, 'min_child_samples': 8.583576452266625, 'min_child_weight': 23.127616337880774, 'feature_pre_filter': False, 'min_data_in_leaf': 45, 'objective': 'binary', 'learning_rate': 0.005, 'bagging_freq': 1, 'force_row_wise': True, 'random_state': 1991}
[LightGBM] [Warning] min_data_in_leaf is set=45, min_child_samples=8.583576452266625 will be ignored. Current value: min_data_in_leaf=45
[LightGBM] [Warning] min_data_in_leaf is set=45, min_child_samples=8.583576452266625 will be ignored. Current value: min_data_in_leaf=45
[LightGBM] [Info] Number of positive: 17383, number of negativ

### 결과 확인

In [ ]:
max_params = optimizer.max['params']
max_params

{'num_leaves': 39.78408573418802,
 'lambda_l1': 0.847247634454523,
 'lambda_l2': 0.9684570594083357,
 'feature_fraction': 0.659585555790575,
 'bagging_fraction': 0.7723146295159312,
 'min_child_samples': 7.8105337253492335,
 'min_child_weight': 25.85681167932419,
 'min_data_in_leaf': 31.07484703764963}

In [ ]:
# 정수형 하이퍼파라미터 변환
max_params['num_leaves'] = int(round(max_params['num_leaves']))
max_params['min_child_samples'] = int(round(max_params['min_child_samples']))
max_params['min_data_in_leaf'] = int(round(max_params['min_data_in_leaf']))

In [ ]:
# 값이 고정된 하이퍼파라미터 추가
max_params.update(fixed_params)
max_params

{'num_leaves': 40,
 'lambda_l1': 0.847247634454523,
 'lambda_l2': 0.9684570594083357,
 'feature_fraction': 0.659585555790575,
 'bagging_fraction': 0.7723146295159312,
 'min_child_samples': 8,
 'min_child_weight': 25.85681167932419,
 'min_data_in_leaf': 31,
 'objective': 'binary',
 'learning_rate': 0.005,
 'bagging_freq': 1,
 'force_row_wise': True,
 'random_state': 1991}

## 모델 훈련 및 성능 검증

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1991)

In [ ]:
oof_val_preds = np.zeros(X.shape[0])
oof_test_preds = np.zeros(X_test.shape[0])

In [ ]:
for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    print('#'*40, f'폴드 {idx+1} / {folds.n_splits}', '#'*40)

    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]

    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_valid, y_valid)

    lgb_model = lgb.train(params=max_params,
                          train_set=dtrain,
                          num_boost_round=2500,
                          valid_sets=dvalid,
                          feval=gini,
                          callbacks=[
                              lgb.early_stopping(stopping_rounds=300),
                              lgb.log_evaluation(100)
                        ])

    oof_test_preds += lgb_model.predict(X_test)/folds.n_splits
    oof_val_preds[valid_idx] += lgb_model.predict(X_valid)

    gini_score = eval_gini(y_valid, oof_val_preds[valid_idx])
    print(f'폴드 {idx+1} 지니계수: {gini_score}/n')

######################################## 폴드 1 / 5 ########################################
[LightGBM] [Warning] min_data_in_leaf is set=31, min_child_samples=8 will be ignored. Current value: min_data_in_leaf=31
[LightGBM] [Warning] min_data_in_leaf is set=31, min_child_samples=8 will be ignored. Current value: min_data_in_leaf=31
[LightGBM] [Info] Number of positive: 17355, number of negative: 458814
[LightGBM] [Info] Total Bins 1554
[LightGBM] [Info] Number of data points in the train set: 476169, number of used features: 216
[LightGBM] [Warning] min_data_in_leaf is set=31, min_child_samples=8 will be ignored. Current value: min_data_in_leaf=31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764
Training until validation scores don't improve for 300 rounds
[100]	valid_0's binary_logloss: 0.154231	valid_0's gini: 0.269234
[200]	valid_0's binary_logloss: 0.153173	valid_0's gini: 0.274691
[300]	valid_0's b

In [ ]:
print('OOF 검증 데이터 지니계수: ', eval_gini(y, oof_val_preds))

OOF 검증 데이터 지니계수:  0.28845355231390746


## 예측 및 결과 제출

In [ ]:
submission['target'] = oof_test_preds
submission.to_csv('submission.csv')